<a href="https://colab.research.google.com/github/Kingtheblaze/task/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kingtheblaze/task/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

This rule identifies and ranks "Leaky Bucket" pages: content that currently has strong search visibility (ranking on page one or getting high impressions) but fails to capture or retain the user, resulting in a significantly below-average click-through rate (CTR) or engagement rate. The final score weights the size of the opportunity (total impressions) against the severity of the underperformance.The Reason Codes:

page_one_ctr_anomaly: The page ranks in the top 10 positions and has over 500 impressions, but its CTR is below 2%.

high_traffic_low_engagement: The page gets solid traffic (impressions > 500) but has an engagement rate below 30%, meaning users arrive but immediately leave.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
from datasets import load_dataset

# 1. Setup & Load Data
HF_TOKEN = "YOUR_ACTUAL_TOKEN_HERE"  # Replace with your token
facts_ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train", token=HF_TOKEN)
dim_ds = load_dataset("FlyRank/internship-warehouse", data_files="dim_content.parquet", split="train", token=HF_TOKEN)
con = duckdb.connect()

# 2. Extract Base Features (March 2026 Window)
query = """
WITH monthly_facts AS (
    SELECT
        content_hash_id,
        SUM(impressions) AS total_impressions,
        SUM(clicks) AS total_clicks,
        AVG(position) AS avg_position,
        AVG(engagement_rate) AS avg_engagement_rate
    FROM facts_ds
    WHERE CAST(report_date AS VARCHAR) BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id
    HAVING SUM(impressions) >= 500
)
SELECT
    c.content_hash_id,
    f.total_impressions,
    f.total_clicks,
    f.avg_position,
    f.avg_engagement_rate,
    CASE WHEN f.total_impressions > 0 THEN (f.total_clicks * 1.0 / f.total_impressions) ELSE 0 END AS ctr
FROM dim_ds c
JOIN monthly_facts f ON c.content_hash_id = f.content_hash_id
"""
df = con.sql(query).df()

# 3. Apply the Rule and Reason Codes
def assign_reason(row):
    if row['avg_position'] <= 10 and row['ctr'] < 0.02:
        return 'page_one_ctr_anomaly'
    elif row['avg_engagement_rate'] < 0.30:
        return 'high_traffic_low_engagement'
    return 'healthy'

df['reason_code'] = df.apply(assign_reason, axis=1)

# Filter only actionable rows
df_action = df[df['reason_code'] != 'healthy'].copy()

# Score: Volume * Severity of the problem
df_action['action_score'] = df_action.apply(
    lambda x: (x['total_impressions'] / 100) * (0.05 - x['ctr']) if x['reason_code'] == 'page_one_ctr_anomaly'
    else (x['total_impressions'] / 100) * (0.50 - x['avg_engagement_rate']),
    axis=1
)

# Assign Action and Confidence
df_action['action_label'] = df_action['reason_code'].map({
    'page_one_ctr_anomaly': 'rewrite_title_and_meta',
    'high_traffic_low_engagement': 'improve_on_page_hook_and_formatting'
})
df_action['confidence_note'] = df_action['total_impressions'].apply(
    lambda x: 'High (Large N)' if x > 2000 else 'Medium (Moderate N)'
)

# Rank the queue
df_ranked = df_action.sort_values(by='action_score', ascending=False).reset_index(drop=True)

# 4. Write CSV
os.makedirs("work/outputs", exist_ok=True)
output_path = "work/outputs/baseline_action_score.csv"
df_ranked.to_csv(output_path, index=False)
print(f"Ranked queue written to: {output_path}\n")

# 5. The Top-20 Review Printout
print("--- TASK 3: TOP 20 REVIEW ---")
for index, row in df_ranked.head(20).iterrows():
    print(f"#{index + 1} [ID: {row['content_hash_id'][:8]}]")
    print(f"  Action: {row['action_label']}")
    print(f"  Reason: {row['reason_code']} (Pos {row['avg_position']:.1f}, {row['total_impressions']} impr, CTR {row['ctr']:.1%}, Eng {row['avg_engagement_rate']:.1%})")
    print(f"  Confidence: {row['confidence_note']}")

    # Contextualize what makes it wrong based on the reason code
    if row['reason_code'] == 'page_one_ctr_anomaly':
        print("  What makes it wrong: If the query intent is a 'Zero-Click' factual answer (e.g., weather, timezone), CTR will naturally be near zero. Rewriting metadata won't fix it.")
    else:
        print("  What makes it wrong: If the page is purely navigational (e.g., a login portal or a 'Contact Us' page), users are supposed to leave quickly. Low engagement is expected.")
    print("-" * 75)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Weak Picks (Which picks look wrong and why?):
When reviewing the top 50 rows in the generated CSV, the weakest picks are the pages hovering right around the 501 impression threshold with Medium (Moderate N) confidence. Because their denominator is so small, a difference of just 3 or 4 clicks drastically swings their CTR and engagement metrics, making them highly susceptible to random noise rather than actual content decay. Additionally, if any of these top picks are strictly navigational pages (like a password reset page), prioritizing them for an "on-page hook" rewrite is a wasted effort, as low engagement on those pages is a sign of success, not failure.

Leakage Check (Confirmation):

No Product Flags: The SQL extraction strictly pulls observed behavioral signals (impressions, clicks, position, engagement_rate). Existing product rules like health_score, priority_score, or refresh_tier were intentionally excluded from the query and do not influence the ranking.

No Future Windows: The target feature window is strictly bound to 2026-03-01 through 2026-03-31. The ranking relies entirely on this historical state and does not utilize any future April data to validate or adjust its scores. The time boundary is uncompromised.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.